In [ ]:
import glob, os, re, json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from scipy.stats import ttest_rel, pearsonr
from scipy.io import loadmat
from IPython.display import display, Math, Latex

band_names = [r"$\delta$",r"$\theta$",r"$\alpha$",r"$\beta_{LOW}$",r"$\beta_{MID}$",r"$\beta_{HIGH}$",r"$\gamma$",r"$\beta_{ALL}$",r"$1-40 Hz$"]

# Amount of non-linearity

## fMRI

In [ ]:
# loading fMRI data
listheo=[]
listrue=[]
fmri_res =  {}
for folder in glob.glob("../../NonLinearityResultsNew/shaved_eso245_cra_strin_*"):
    if os.path.isfile(os.path.join(folder,os.path.split(folder)[1]+"_globalStats.json")):
        statsMI = np.load(os.path.join(folder,f"subject00_8.npy"))
        pairs = statsMI.shape[0]
        regions = int(0.5+np.sqrt(0.25+2*pairs))
        theo = int(re.findall(r"_(\d+)", folder)[0])
        listheo.append(theo)
        listrue.append(regions)
        print(folder, theo, regions, statsMI.shape)
        with open(os.path.join(folder,os.path.split(folder)[1]+"_globalStats.json")) as fp:
            fmri_res[regions] = {k:np.array(v) for k, v in json.load(fp).items()}
listheo.append(18380)
listrue.append(18380)
tru2theo = {k:v for k, v in zip(listrue, listheo)}
theo2tru = {v:k for k, v in zip(listrue, listheo)}
with open("../../NonLinearityResultsNew/shaved_100center_bin8/shaved_100center_bin8_globalStats.json") as fp:
    fmri_res[18380] = {k:np.array(v) for k, v in json.load(fp).items()}
#del statsReg[586]
order = np.argsort(listheo)
plt.plot(np.array(listheo)[order], np.array(listrue)[order], "o--")
plt.plot(np.array(listheo)[order], np.array(listheo)[order], ":k", lw=0.5)
plt.xlabel("Designed number of regions")
plt.ylabel("Effective number of regions")
plt.xscale("log")
plt.yscale("log")
plt.show()

In [ ]:
for i in fmri_res:
    fmri_res[i]["RNL"] = (fmri_res[i]["totalMI"]-fmri_res[i]["gaussMI"])/fmri_res[i]["totalMI"]
    fmri_res[i]["RNLshadow"] = (fmri_res[i]["totalMIshadow"]-fmri_res[i]["gaussMIshadow"])/fmri_res[i]["totalMIshadow"]

In [ ]:
fig = plt.figure(figsize=(9,5))
x=sorted(fmri_res.keys())#[theo2tru[k] for k in sorted(theo2tru.keys())]#
STRETCH=1.8

trueRNL = [fmri_res[i]["RNL"] for i in x]
plt.boxplot(trueRNL, positions=np.arange(len(x))*STRETCH-0.3,boxprops={"color":"darkorange"}, whiskerprops={"color":"darkorange"}, flierprops={"markeredgecolor":"darkorange", "marker":"+"}, medianprops={"lw":1.6,"color":"darkorange"}, capprops={"color":"darkorange"})
shadowRNL = [fmri_res[i]["RNLshadow"] for i in x]
plt.boxplot(shadowRNL, positions=np.arange(len(x))*STRETCH+0.3,boxprops={"color":"blue"}, whiskerprops={"color":"blue"}, flierprops={"markeredgecolor":"blue", "marker":"+"}, medianprops={"lw":1.6,"color":"blue"}, capprops={"color":"blue"})

tkpos=np.arange(len(x))*STRETCH+0
tickNames = [str(n) for n in sorted(fmri_res.keys())]#sorted(theo2tru.keys())]
tickNames[-1] = "Sing.\nVoxel"
plt.xticks(tkpos, tickNames, rotation=60, fontsize="small")

p1,= plt.plot([-0.5, len(x)*STRETCH-1], [0,0], "-r", lw=1, label=0, zorder=1)
p2= plt.hlines(0.1,-0.5, len(x)*STRETCH-1, "r", "dashed", lw=1, label="10%", zorder=1)
plt.ylabel("Relative non-linearity in MI")
plt.xlabel("# of regions")
plt.legend([Patch(facecolor="white", edgecolor="darkorange"), Patch(facecolor="white", edgecolor="blue"), p1, p2], ["Total MI vs Gauss MI", "Shadow dataset", r"0%", r"10%"])
plt.title(f"{len(fmri_res[10]['totalMI'])} subjects $-$ rs-fMRI")
# plt.savefig("xxxxfMRISha.pdf", bbox_inches="tight")
#plt.ylim((-0.05, 0.15))
plt.show()

In [ ]:
sig_diff_reg = np.array([ttest_rel(T,S).pvalue for T, S in zip(trueRNL, shadowRNL)])
avTrueRNL_fMRI = np.mean(trueRNL, axis=1)
avShadRNL_fMRI = np.mean(shadowRNL, axis=1)
palette = plt.cm.magma
palette.set_bad(color="black") # Bad values (i.e., masked, set to black 0.8
palette.set_under(color="grey") # Bad values (i.e., masked, set to black 0.8
palette.set_over(color="yellow") # Bad values (i.e., masked, set to black 0.8
A = np.array(sorted(fmri_res.keys()), dtype=float)#sorted(theo2tru.keys())

fig, ax = plt.subplots(1,2,figsize=(9,5))
plt.sca(ax[0])
#A[sig_diff_reg>0.05]=np.nan
plt.scatter(avTrueRNL_fMRI, avShadRNL_fMRI,c=A,cmap=palette, vmax=1000, vmin=15)
cb = plt.colorbar(extend="max")
cb.ax.set_ylabel('# regions', rotation=90)
# plt.plot([max(min(avTrueRNL_fMRI),min(avShadRNL_fMRI))]*2, [min(max(avTrueRNL_fMRI),max(avShadRNL_fMRI))]*2, ":k", lw=1)
plt.xlabel("Data RNL")
plt.ylabel("Shadow RNL")

plt.sca(ax[1])
plt.scatter(A, avTrueRNL_fMRI, label="True")
plt.scatter(A, avShadRNL_fMRI, label="Shadow")
plt.scatter(A, avTrueRNL_fMRI-avShadRNL_fMRI, label="Difference", marker="+")
plt.xscale("log")
plt.xlabel('# regions')
plt.ylabel('RNL')
plt.legend(loc=0)
ax[1].yaxis.set_label_position("right")
ax[1].yaxis.tick_right()
print(A[np.argmax(avTrueRNL_fMRI-avShadRNL_fMRI)])
plt.suptitle("Relationship between true and shadow $-$ rs-fMRI")
plt.show()

In [ ]:
palette = plt.cm.magma
palette.set_bad(color="black") # Bad values (i.e., masked, set to black 0.8
palette.set_under(color="grey") # Bad values (i.e., masked, set to black 0.8
palette.set_over(color="yellow") # Bad values (i.e., masked, set to black 0.8
A = np.array(sorted(fmri_res.keys()), dtype=float)#sorted(theo2tru.keys())

fig, ax = plt.subplots(1,1,figsize=(9,9))
plt.sca(ax)
#A[sig_diff_reg>0.05]=np.nan
for i in range(24):
    plt.scatter(trueRNL[i], shadowRNL[i],c=np.ones(242)*A[i],cmap=palette, vmax=1000, vmin=15, s=5)
ax.set_aspect(1)
pts = [max(plt.xlim()[0], plt.ylim()[0]),min(plt.xlim()[1], plt.ylim()[1])]
plt.xlim(plt.xlim())
plt.ylim(plt.ylim())
plt.plot(pts, pts, ":k", lw=1)
plt.xlabel("Data RNL")
plt.ylabel("Shadow RNL")
plt.show()

palette.set_over(color="grey") # Bad values (i.e., masked, set to black 0.8
corr_fmri = [pearsonr(trueRNL[i], shadowRNL[i]) for i in range(24)]
plt.scatter(A, [q[0] for q in corr_fmri],c=[q[1] for q in corr_fmri],cmap=palette, vmax=0.05, vmin=0)
plt.xscale("log")
plt.show()


## EEG

**Fixed time** corresponds to *45 s*

**Fixed samples** is *1000 samples*

In particular delta and theta fixed time bands have less than 1000 samples (405, 810).

In [ ]:
for b in range(1,10):
    mat = loadmat(f"../../NonLinearityData/EEG_el_so_BLP_NEW/NEW_EEG_fixedTime_band{b}.mat")['EEG']
    display(Latex(band_names[b-1]+f" = {mat.shape[0]}"))

### Scalp Voltage

In [ ]:
eeg_scalp_volt_Samp = {}
for folder in glob.glob("../../NonLinearityResultsNew/NEW_EEG_fixedSamples_band?_bin10"):
    if os.path.isfile(os.path.join(folder,os.path.split(folder)[1]+"_globalStats.json")):
        try:
            statsMI = np.load(os.path.join(folder,f"subject00_10.npy"))
        except:
            statsMI=np.array([])
        pairs = statsMI.shape[0]
        band = int(re.findall(r"band(\d+)", folder)[0])
        print(folder, band, statsMI.shape)
        with open(os.path.join(folder,os.path.split(folder)[1]+"_globalStats.json")) as fp:
            eeg_scalp_volt_Samp[band] = {k:np.array(v) for k, v in json.load(fp).items()}

eeg_scalp_volt_Time = {}
for folder in glob.glob("../../NonLinearityResultsNew/NEW_EEG_fixedTime_band?_bin*"):
    bins = int(re.findall("bin(\d+)", folder)[0])
    if os.path.isfile(os.path.join(folder,os.path.split(folder)[1]+"_globalStats.json")):
        try:
            statsMI = np.load(os.path.join(folder,f"subject00_{bins}.npy"))
        except:
            statsMI=np.array([])
        pairs = statsMI.shape[0]
        band = int(re.findall(r"band(\d+)", folder)[0])
        print(folder, band, statsMI.shape)
        with open(os.path.join(folder,os.path.split(folder)[1]+"_globalStats.json")) as fp:
            eeg_scalp_volt_Time[band] = {k:np.array(v) for k, v in json.load(fp).items()}

In [ ]:
for i in eeg_scalp_volt_Samp:
    eeg_scalp_volt_Samp[i]["RNL"] = (eeg_scalp_volt_Samp[i]["totalMI"]-eeg_scalp_volt_Samp[i]["gaussMI"])/eeg_scalp_volt_Samp[i]["totalMI"]
    eeg_scalp_volt_Samp[i]["RNLshadow"] = (eeg_scalp_volt_Samp[i]["totalMIshadow"]-eeg_scalp_volt_Samp[i]["gaussMIshadow"])/eeg_scalp_volt_Samp[i]["totalMIshadow"]
    eeg_scalp_volt_Time[i]["RNL"] = (eeg_scalp_volt_Time[i]["totalMI"]-eeg_scalp_volt_Time[i]["gaussMI"])/eeg_scalp_volt_Time[i]["totalMI"]
    eeg_scalp_volt_Time[i]["RNLshadow"] = (eeg_scalp_volt_Time[i]["totalMIshadow"]-eeg_scalp_volt_Time[i]["gaussMIshadow"])/eeg_scalp_volt_Time[i]["totalMIshadow"]

In [ ]:
STRETCH = 1.5
x=sorted(eeg_scalp_volt_Samp.keys())
basepos = np.arange(len(x))*STRETCH
fig, ax = plt.subplots(1,2,figsize=(9,5), sharey=True)

for i, extra in enumerate(["","shadow"]):
    plt.sca(ax[i])
    pos = basepos-0.33
    y=[eeg_scalp_volt_Time[i]["RNL"+extra] for i in x]
    plt.boxplot(y,positions=pos, widths=0.6,boxprops={"color":"blue"}, whiskerprops={"color":"blue"}, flierprops={"markeredgecolor":"blue", "marker":"+"}, medianprops={"lw":1.6,"color":"blue"}, capprops={"color":"blue"})

    pos = basepos+0.33
    y=[eeg_scalp_volt_Samp[i]["RNL"+extra] for i in x]
    plt.boxplot(y,positions=pos, widths=0.6,boxprops={"color":"darkorange"}, whiskerprops={"color":"darkorange"}, flierprops={"markeredgecolor":"darkorange", "marker":"+"}, medianprops={"lw":1.6,"color":"darkorange"}, capprops={"color":"darkorange"})

    p1= plt.hlines(0.,basepos[0]-0.6, basepos[-1]+0.6, "r", "solid", lw=1, label="%", zorder=1)
    p2= plt.hlines(0.1,basepos[0]-0.6, basepos[-1]+0.6, "r", "dotted", lw=1, label="10%", zorder=1)

    plt.ylabel(f"relative difference in {extra} MI")
    plt.xlabel("Band")
    plt.xticks(ticks=basepos, labels=band_names)
    plt.title("Shadow" if extra else "True")
ax[1].yaxis.set_label_position("right")
ax[1].yaxis.tick_right()
plt.legend([Patch(facecolor="white", edgecolor="blue"), Patch(facecolor="white", edgecolor="darkorange"), p1, p2], ["Fixed time", "Fixed #samples", r"0", r"10%"], loc=0)
plt.suptitle("50 subjects $-$ Scalp Voltage")
plt.show()

In [ ]:
#tab20
avTrueRNL_EEG_scalp_volt_samp = np.mean([eeg_scalp_volt_Samp[i]["RNL"] for i in x], axis=1)
avShadRNL_EEG_scalp_volt_samp = np.mean([eeg_scalp_volt_Samp[i]["RNLshadow"] for i in x], axis=1)
avTrueRNL_EEG_scalp_volt_time = np.mean([eeg_scalp_volt_Time[i]["RNL"] for i in x], axis=1)
avShadRNL_EEG_scalp_volt_time = np.mean([eeg_scalp_volt_Time[i]["RNLshadow"] for i in x], axis=1)
palette = ListedColormap(plt.cm.tab20.colors[:-2])

fig, ax = plt.subplots(1,2,figsize=(9,5))
plt.sca(ax[0])
#A[sig_diff_reg>0.05]=np.nan
plt.scatter(avTrueRNL_EEG_scalp_volt_samp, avShadRNL_EEG_scalp_volt_samp,c=np.arange(0,18,2),cmap=palette, vmax=18, vmin=0, label="Fixed Samples")
plt.scatter(avTrueRNL_EEG_scalp_volt_time, avShadRNL_EEG_scalp_volt_time,c=np.arange(1,18,2),cmap=palette, vmax=18, vmin=0, label="Fixed Time")
cb = plt.colorbar()
cb.ax.set_ylabel('# regions', rotation=90)
cb.ax.get_yaxis().set_ticks(np.arange(1,18,2), band_names)
pts = [max(plt.xlim()[0], plt.ylim()[0]),min(plt.xlim()[1], plt.ylim()[1])]
plt.xlim(plt.xlim())
plt.ylim(plt.ylim())
plt.plot(pts, pts, ":k", lw=1)
plt.xlabel("Data RNL")
plt.ylabel("Shadow RNL")
plt.legend(loc=0)

plt.sca(ax[1])
plt.scatter(np.arange(0,9), avTrueRNL_EEG_scalp_volt_samp, color=plt.cm.tab20.colors[0], label="True")
plt.scatter(np.arange(0,9), avShadRNL_EEG_scalp_volt_samp, color=plt.cm.tab20.colors[2], label="Shadow")
plt.scatter(np.arange(0,9), avTrueRNL_EEG_scalp_volt_samp-avShadRNL_EEG_scalp_volt_samp, color=plt.cm.tab20.colors[4], label="Difference", marker="+")
plt.scatter(np.arange(0,9), avTrueRNL_EEG_scalp_volt_time, color=plt.cm.tab20.colors[1], label="True")
plt.scatter(np.arange(0,9), avShadRNL_EEG_scalp_volt_time, color=plt.cm.tab20.colors[3], label="Shadow")
plt.scatter(np.arange(0,9), avTrueRNL_EEG_scalp_volt_time-avShadRNL_EEG_scalp_volt_time, color=plt.cm.tab20.colors[5], label="Difference", marker="+")
# plt.xscale("log")
plt.xlabel('# regions')
plt.ylabel('RNL')
plt.legend(loc=0)
plt.xticks(np.arange(0,9),band_names)
ax[1].yaxis.set_label_position("right")
ax[1].yaxis.tick_right()
plt.suptitle("Relationship between true and shadow $-$ Scalp Voltage")
plt.show()

In [ ]:
palette = ListedColormap(plt.cm.tab20.colors[:-2])

fig, ax = plt.subplots(1,2,figsize=(13,7))
for ax_n, pack in enumerate([("Samples", eeg_scalp_volt_Samp), ("Time", eeg_scalp_volt_Time)]):
    kind, data = pack
    plt.sca(ax[ax_n])
    for i in range(1,10):
        plt.scatter(data[i]["RNL"], data[i]["RNLshadow"],c=np.full(50, i*2+(kind=="Time")),cmap=palette, vmax=18, vmin=0, s=5)
    ax[ax_n].set_aspect(1)
    pts = [max(plt.xlim()[0], plt.ylim()[0]),min(plt.xlim()[1], plt.ylim()[1])]
    plt.xlim(plt.xlim())
    plt.ylim(plt.ylim())
    plt.plot(pts, pts, ":k", lw=1)
    plt.xlabel("Data RNL")
    plt.ylabel("Shadow RNL")
    plt.title(kind)
plt.show()

palette = plt.cm.magma
palette.set_bad(color="black") # Bad values (i.e., masked, set to black 0.8
palette.set_under(color="grey") # Bad values (i.e., masked, set to black 0.8
palette.set_over(color="grey") # Bad values (i.e., masked, set to black 0.8
for ax_n, pack in enumerate([("Samples", eeg_scalp_volt_Samp, "o"), ("Time", eeg_scalp_volt_Time, "^")]):
    kind, data, marker = pack
    corr_eeg = [pearsonr(data[i]["RNL"], data[i]["RNLshadow"]) for i in range(1,10)]
    plt.scatter(np.arange(0,9), [q[0] for q in corr_eeg],c=[q[1] for q in corr_eeg],cmap=palette, vmax=0.05, vmin=0, marker=marker, label=kind)
plt.xticks(np.arange(0,9),band_names)
plt.legend(loc=0)
plt.show()

### Scalp BLP

In [ ]:
eeg_scalp_blp = {k:{s:[{} for b in range(9)] for s in ["Shadow", "Normal"]} for k in ["Time", "Samples"]}  #eeg_scalp_blp[kind][shadow][band-1][avg]
for folder in glob.glob("../../NonLinearityResultsNew/NEW_electrodeBLP_fixed*_avg*_band?_bin*"):
    if os.path.isfile(os.path.join(folder,os.path.split(folder)[1]+"_globalStats.json")):
        band = int(re.findall(r"band(\d+)", folder)[0])-1
        avg = int(re.findall(r"avg(\d+)", folder)[0])
        kind = "Time" if "Time" in folder else "Samples"
        bins = int(re.findall(r"bin(\d+)", folder)[0])
        shadow = "Shadow" if "shadow" in folder else "Normal"
        try:
            statsMI = np.load(os.path.join(folder,f"subject00_{bins}.npy"))
        except:
            statsMI=np.array([])
        pairs = statsMI.shape[0]
        print(folder, band, statsMI.shape)
        with open(os.path.join(folder,os.path.split(folder)[1]+"_globalStats.json")) as fp:
            eeg_scalp_blp[kind][shadow][band][avg] = {k:np.array(v) for k, v in json.load(fp).items()}

In [ ]:
for kind in ["Time", "Samples"]:
    for shadow in ["Shadow", "Normal"]:
        for band in range(9):
            for avg in eeg_scalp_blp[kind][shadow][band]:
                eeg_scalp_blp[kind][shadow][band][avg]["RNL"] = 1- eeg_scalp_blp[kind][shadow][band][avg]["gaussMI"]/eeg_scalp_blp[kind][shadow][band][avg]["totalMI"]
                if shadow == "Normal":
                    eeg_scalp_blp[kind][shadow][band][avg]["RNLshadow"] = 1- eeg_scalp_blp[kind][shadow][band][avg]["gaussMIshadow"]/eeg_scalp_blp[kind][shadow][band][avg]["totalMIshadow"]

In [ ]:
colors = {"Time":"darkorange", "Samples":"b"}
STRETCH = 3
basepos = np.arange(len(band_names))*STRETCH
fig, ax = plt.subplots(1,3,figsize=(15,5), sharey=True)

for ax_n, pack in enumerate([("True", "Normal", ""), ("Shadow After", "Normal", "shadow"), ("Shadow Before", "Shadow", "")]):
    plt.sca(ax[ax_n])
    title, shadow, sub = pack
    offs = -2.5
    for avg in (1,2,4):
        for kind in ("Time", "Samples"):
            color = colors[kind] if avg<4 else "magenta"
            if avg in eeg_scalp_blp[kind][shadow][0]:
                y=[eeg_scalp_blp[kind][shadow][band][avg]["RNL"+sub] for band in range(9)]
                plt.boxplot(y,positions=np.arange(9)*STRETCH+offs, boxprops={"color":color}, whiskerprops={"color":color}, flierprops={"markeredgecolor":color, "marker":"+"}, medianprops={"lw":1.6,"color":color}, capprops={"color":color})
                offs += 0.5

    p1= plt.hlines(0.,-3, (len(band_names)-1)*STRETCH, "r", "solid", lw=1, label="%", zorder=1)
    p2= plt.hlines(0.1,-3, (len(band_names)-1)*STRETCH, "r", "dashed", lw=1, label="10%", zorder=1)
    tkpos=np.arange(9)+1
    plt.ylabel("relative difference in MI")
    plt.xlabel("Band")
    plt.xticks(ticks=np.arange(9)*STRETCH-1.5, labels=band_names)
    plt.title(title)
plt.legend([Patch(facecolor="white", edgecolor="darkorange"),Patch(facecolor="white", edgecolor="b"), Patch(facecolor="white", edgecolor="magenta"), p1, p2], ["Fixed Time Len", "Fixed #Samples", "Both", r"0", r"10%"])
plt.suptitle("50 subjects $-$ Scalp BLP")
plt.show()

In [ ]:
#tab20
avTrueRNL_EEG_scalp_blp = {kind:{avg:np.mean([eeg_scalp_blp[kind]["Normal"][band][avg]["RNL"]for band in range(9)], axis=1) for avg in eeg_scalp_blp[kind]["Normal"][0]} for kind in ["Time", "Samples"]}
avShadRNL_EEG_scalp_blp = {kind:{avg:np.mean([eeg_scalp_blp[kind]["Shadow"][band][avg]["RNL"]for band in range(9)], axis=1) for avg in eeg_scalp_blp[kind]["Shadow"][0]} for kind in ["Time", "Samples"]}
palette = ListedColormap(plt.cm.tab20.colors[:-2])

fig, ax = plt.subplots(1,2,figsize=(9,5))
plt.sca(ax[0])
for kind in ["Samples", "Time"]:
    for av, marker in zip([1,2,4],"o*^"):
        if av in avTrueRNL_EEG_scalp_blp[kind]:
            plt.scatter(avTrueRNL_EEG_scalp_blp[kind][av], avShadRNL_EEG_scalp_blp[kind][av],c=np.arange(0,18,2)+(kind=="Time"),cmap=palette, vmax=18, vmin=0, marker=marker)
cb = plt.colorbar()
cb.ax.set_ylabel('Band', rotation=90)
cb.ax.get_yaxis().set_ticks(np.arange(1,18,2), band_names)
pts = [max(plt.xlim()[0], plt.ylim()[0]),min(plt.xlim()[1], plt.ylim()[1])]
plt.xlim(plt.xlim())
plt.ylim(plt.ylim())
plt.plot(pts, pts, ":k", lw=1)
plt.xlabel("Data RNL")
plt.ylabel("Shadow RNL")
ha = [Line2D([],[],lw=0,color=c,marker="o") for c in ["black", "grey"]]
ha += [Line2D([],[],lw=0,color="k",marker=m) for m in "o*^"] 
la = ["Fixed Samples", "Fixed Time", "125 ms", "250 ms", "500 ms", ]
plt.legend(ha, la, loc="upper left", frameon=False)

plt.sca(ax[1])
for kind in ["Samples", "Time"]:
    offset=-0.28
    for av, marker in zip([1,2,4],"o*^"):
        if av in avTrueRNL_EEG_scalp_blp[kind]:
            plt.scatter(np.arange(0,9)+offset, avTrueRNL_EEG_scalp_blp[kind][av], color=plt.cm.tab20.colors[0+(kind=="Time")], marker=marker)
            plt.scatter(np.arange(0,9)+offset, avShadRNL_EEG_scalp_blp[kind][av], color=plt.cm.tab20.colors[2+(kind=="Time")], marker=marker)
            plt.scatter(np.arange(0,9)+offset, avTrueRNL_EEG_scalp_blp[kind][av]-avShadRNL_EEG_scalp_blp[kind][av], color=plt.cm.tab20.colors[4+(kind=="Time")], marker=marker)
        offset+=0.28
# plt.xscale("log")
plt.xlabel('Band')
plt.ylabel('RNL')
ha = [Line2D([],[],lw=0,color="k",marker=m) for m in "o*^"] 
ha += [Line2D([],[],lw=0,color=plt.cm.tab20.colors[i],marker="o") for i in [0,2,4]] 
ha += [Line2D([],[],lw=0,color=c,marker="o") for c in ["black", "grey"]]
la = ["125 ms", "250 ms", "500 ms", "True", "Shadow", "Difference", "Fixed Samples", "Fixed Time"]
plt.legend(ha, la, bbox_to_anchor=(1.2, 0.5), loc="center left", frameon=False)
plt.xticks(np.arange(0,9),band_names)
ax[1].yaxis.set_label_position("right")
ax[1].yaxis.tick_right()
plt.suptitle("Relationship between true and shadow $-$ Scalp BLP")
plt.show()

In [ ]:
palette = ListedColormap(plt.cm.tab20.colors[:-2])

fig, ax = plt.subplots(1,2,figsize=(13,7))
for ax_n, kind in enumerate(["Samples", "Time"]):
    plt.sca(ax[ax_n])
    for i in range(9):
        for avg, marker in zip(eeg_scalp_blp[kind]["Normal"][i],"o*^"):
            plt.scatter(eeg_scalp_blp[kind]["Normal"][i][avg]["RNL"], eeg_scalp_blp[kind]["Shadow"][i][avg]["RNL"],c=np.full(50, i*2+(kind=="Time")),cmap=palette, vmax=18, vmin=0, s=5, marker=marker)
    ax[ax_n].set_aspect(1)
    pts = [max(plt.xlim()[0], plt.ylim()[0]),min(plt.xlim()[1], plt.ylim()[1])]
    plt.xlim(plt.xlim())
    plt.ylim(plt.ylim())
    plt.plot(pts, pts, ":k", lw=1)
    plt.xlabel("Data RNL")
    plt.ylabel("Shadow RNL")
    plt.title(kind)
plt.show()


palette = plt.cm.magma
palette.set_bad(color="black") # Bad values (i.e., masked, set to black 0.8
palette.set_under(color="grey") # Bad values (i.e., masked, set to black 0.8
palette.set_over(color="grey") # Bad values (i.e., masked, set to black 0.8
for kind, marker in [("Samples", "o"), ("Time", "^")]:
    for avg, s in zip(eeg_scalp_blp[kind]["Normal"][0],[5,10,20]):
        corr_ieeg = [pearsonr(eeg_scalp_blp[kind]["Normal"][i][avg]["RNL"], eeg_scalp_blp[kind]["Shadow"][i][avg]["RNL"]) for i in range(9)]
        plt.scatter(np.arange(0,9), [q[0] for q in corr_ieeg],c=[q[1] for q in corr_ieeg],cmap=palette, vmax=0.05, vmin=0, marker=marker, label=kind if avg==2 else None, s=s)
plt.xticks(np.arange(0,9),band_names)
plt.legend(loc=0)
plt.show()

### Source BLP

In [ ]:
eeg_source_blp = {k:{s:[{} for b in range(9)] for s in ["Shadow", "Normal"]} for k in ["Time", "Samples"]}  #eeg_source_blp[kind][shadow][band-1][avg]
for folder in glob.glob("../../NonLinearityResultsNew/PCA_sourceBLP_fixed*_avg*_band?_bin*"):
    if os.path.isfile(os.path.join(folder,os.path.split(folder)[1]+"_globalStats.json")):
        band = int(re.findall(r"band(\d+)", folder)[0])-1
        avg = int(re.findall(r"avg(\d+)", folder)[0])
        kind = "Time" if "Time" in folder else "Samples"
        bins = int(re.findall(r"bin(\d+)", folder)[0])
        shadow = "Shadow" if "shadow" in folder else "Normal"
        try:
            assert False
            statsMI = np.load(os.path.join(folder,f"subject00_{bins}.npy"))
        except:
            statsMI=np.array([])
        print(folder, band, statsMI.shape)
        with open(os.path.join(folder,os.path.split(folder)[1]+"_globalStats.json")) as fp:
            eeg_source_blp[kind][shadow][band][avg] = {k:np.array(v) for k, v in json.load(fp).items()}

In [ ]:
for kind in ["Time", "Samples"]:
    for shadow in ["Shadow", "Normal"]:
        for band in range(9):
            for avg in eeg_source_blp[kind][shadow][band]:
                eeg_source_blp[kind][shadow][band][avg]["RNL"] = 1- eeg_source_blp[kind][shadow][band][avg]["gaussMI"]/eeg_source_blp[kind][shadow][band][avg]["totalMI"]
                if shadow == "Normal":
                    eeg_source_blp[kind][shadow][band][avg]["RNLshadow"] = 1- eeg_source_blp[kind][shadow][band][avg]["gaussMIshadow"]/eeg_source_blp[kind][shadow][band][avg]["totalMIshadow"]

In [ ]:
colors = {"Time":"darkorange", "Samples":"b"}
STRETCH = 3
basepos = np.arange(len(band_names))*STRETCH
fig, ax = plt.subplots(1,3,figsize=(15,5), sharey=True)

for ax_n, pack in enumerate([("True", "Normal", ""), ("Shadow After", "Normal", "shadow"), ("Shadow Before", "Shadow", "")]):
    plt.sca(ax[ax_n])
    title, shadow, sub = pack
    offs = -2.5
    for avg in (1,2,4):
        for kind in ("Time", "Samples"):
            color = colors[kind] if avg<4 else "magenta"
            if avg in eeg_source_blp[kind][shadow][0]:
                y=[eeg_source_blp[kind][shadow][band][avg]["RNL"+sub] for band in range(9)]
                plt.boxplot(y,positions=np.arange(9)*STRETCH+offs, boxprops={"color":color}, whiskerprops={"color":color}, flierprops={"markeredgecolor":color, "marker":"+"}, medianprops={"lw":1.6,"color":color}, capprops={"color":color})
                offs += 0.5

    p1= plt.hlines(0.,-3, (len(band_names)-1)*STRETCH, "r", "solid", lw=1, label="%", zorder=1)
    p2= plt.hlines(0.1,-3, (len(band_names)-1)*STRETCH, "r", "dashed", lw=1, label="10%", zorder=1)
    tkpos=np.arange(9)+1
    plt.ylabel("relative difference in MI")
    plt.xlabel("Band")
    plt.xticks(ticks=np.arange(9)*STRETCH-1.5, labels=band_names)
    plt.title(title)
plt.legend([Patch(facecolor="white", edgecolor="darkorange"),Patch(facecolor="white", edgecolor="b"), Patch(facecolor="white", edgecolor="magenta"), p1, p2], ["Fixed Time Len", "Fixed #Samples", "Both", r"0", r"10%"])
plt.suptitle("50 subjects $-$ Source BLP")
plt.show()

In [ ]:
#tab20
avTrueRNL_EEG_source_blp = {kind:{avg:np.mean([eeg_source_blp[kind]["Normal"][band][avg]["RNL"]for band in range(9)], axis=1) for avg in eeg_source_blp[kind]["Normal"][0]} for kind in ["Time", "Samples"]}
avShadRNL_EEG_source_blp = {kind:{avg:np.mean([eeg_source_blp[kind]["Shadow"][band][avg]["RNL"]for band in range(9)], axis=1) for avg in eeg_source_blp[kind]["Shadow"][0]} for kind in ["Time", "Samples"]}
palette = ListedColormap(plt.cm.tab20.colors[:-2])

fig, ax = plt.subplots(1,2,figsize=(9,5))
plt.sca(ax[0])
for kind in ["Samples", "Time"]:
    for av, marker in zip([1,2,4],"o*^"):
        if av in avTrueRNL_EEG_source_blp[kind]:
            plt.scatter(avTrueRNL_EEG_source_blp[kind][av], avShadRNL_EEG_source_blp[kind][av],c=np.arange(0,18,2)+(kind=="Time"),cmap=palette, vmax=18, vmin=0, marker=marker)
cb = plt.colorbar()
cb.ax.set_ylabel('Band', rotation=90)
cb.ax.get_yaxis().set_ticks(np.arange(1,18,2), band_names)
pts = [max(plt.xlim()[0], plt.ylim()[0]),min(plt.xlim()[1], plt.ylim()[1])]
plt.xlim(plt.xlim())
plt.ylim(plt.ylim())
plt.plot(pts, pts, ":k", lw=1)
plt.xlabel("Data RNL")
plt.ylabel("Shadow RNL")
ha = [Line2D([],[],lw=0,color=c,marker="o") for c in ["black", "grey"]]
ha += [Line2D([],[],lw=0,color="k",marker=m) for m in "o*^"] 
la = ["Fixed Samples", "Fixed Time", "125 ms", "250 ms", "500 ms", ]
plt.legend(ha, la, loc="upper left", frameon=False)

plt.sca(ax[1])
for kind in ["Samples", "Time"]:
    offset=-0.28
    for av, marker in zip([1,2,4],"o*^"):
        if av in avTrueRNL_EEG_source_blp[kind]:
            plt.scatter(np.arange(0,9)+offset, avTrueRNL_EEG_source_blp[kind][av], color=plt.cm.tab20.colors[0+(kind=="Time")], marker=marker)
            plt.scatter(np.arange(0,9)+offset, avShadRNL_EEG_source_blp[kind][av], color=plt.cm.tab20.colors[2+(kind=="Time")], marker=marker)
            plt.scatter(np.arange(0,9)+offset, avTrueRNL_EEG_source_blp[kind][av]-avShadRNL_EEG_source_blp[kind][av], color=plt.cm.tab20.colors[4+(kind=="Time")], marker=marker)
        offset+=0.28
# plt.xscale("log")
plt.xlabel('Band')
plt.ylabel('RNL')
ha = [Line2D([],[],lw=0,color="k",marker=m) for m in "o*^"] 
ha += [Line2D([],[],lw=0,color=plt.cm.tab20.colors[i],marker="o") for i in [0,2,4]] 
ha += [Line2D([],[],lw=0,color=c,marker="o") for c in ["black", "grey"]]
la = ["125 ms", "250 ms", "500 ms", "True", "Shadow", "Difference", "Fixed Samples", "Fixed Time"]
plt.legend(ha, la, bbox_to_anchor=(1.2, 0.5), loc="center left", frameon=False)
plt.xticks(np.arange(0,9),band_names)
ax[1].yaxis.set_label_position("right")
ax[1].yaxis.tick_right()
plt.suptitle("Relationship between true and shadow $-$ Source BLP")
plt.show()

In [ ]:
palette = ListedColormap(plt.cm.tab20.colors[:-2])

fig, ax = plt.subplots(1,2,figsize=(13,7))
for ax_n, kind in enumerate(["Samples", "Time"]):
    plt.sca(ax[ax_n])
    for i in range(9):
        for avg, marker in zip(eeg_source_blp[kind]["Normal"][i],"o*^"):
            plt.scatter(eeg_source_blp[kind]["Normal"][i][avg]["RNL"], eeg_source_blp[kind]["Shadow"][i][avg]["RNL"],c=np.full(50, i*2+(kind=="Time")),cmap=palette, vmax=18, vmin=0, s=5, marker=marker)
    ax[ax_n].set_aspect(1)
    pts = [max(plt.xlim()[0], plt.ylim()[0]),min(plt.xlim()[1], plt.ylim()[1])]
    plt.xlim(plt.xlim())
    plt.ylim(plt.ylim())
    plt.plot(pts, pts, ":k", lw=1)
    plt.xlabel("Data RNL")
    plt.ylabel("Shadow RNL")
    plt.title(kind)
plt.show()


palette = plt.cm.magma
palette.set_bad(color="black") # Bad values (i.e., masked, set to black 0.8
palette.set_under(color="grey") # Bad values (i.e., masked, set to black 0.8
palette.set_over(color="grey") # Bad values (i.e., masked, set to black 0.8
for kind, marker in [("Samples", "o"), ("Time", "^")]:
    for avg, s in zip(eeg_source_blp[kind]["Normal"][0],[5,10,20]):
        corr_ieeg = [pearsonr(eeg_source_blp[kind]["Normal"][i][avg]["RNL"], eeg_source_blp[kind]["Shadow"][i][avg]["RNL"]) for i in range(9)]
        plt.scatter(np.arange(0,9), [q[0] for q in corr_ieeg],c=[q[1] for q in corr_ieeg],cmap=palette, vmax=0.05, vmin=0, marker=marker, label=kind if avg==2 else None, s=s)
plt.xticks(np.arange(0,9),band_names)
plt.legend(loc=0)
plt.show()

In [ ]:
for kind in ["Samples", "Time"]:
    for band in range(9):
        for avg in eeg_source_blp[kind]["Normal"][band]:
            print(kind, band_names[band][2:9], 125*avg, pearsonr(eeg_source_blp[kind]["Normal"][band][avg]["RNL"],eeg_scalp_blp[kind]["Normal"][band][avg]["RNL"])[0], sep="\t")

## iEEG

**Fixed time** corresponds to *45 s*

**Fixed samples** is *1000 samples*

In particular delta and theta fixed time bands have less than 1000 samples (405, 810).

In [ ]:
for b in range(1,10):
    mat = loadmat(f"../../NonLinearityData/iEEG/FIX_iEEG_fixedTime_band{b}.mat")['EEG']
    display(Latex(band_names[b-1]+f" = {mat.shape[0]}"))

In [ ]:
ieeg_volt_Samp = {}
for folder in glob.glob("../../NonLinearityResultsNew/FIX_iEEG_fixedSamples_band?_bin10"):
    if os.path.isfile(os.path.join(folder,os.path.split(folder)[1]+"_globalStats.json")):
        try:
            statsMI = np.load(os.path.join(folder,f"subject00_10.npy"))
        except:
            statsMI=np.array([])
        pairs = statsMI.shape[0]
        band = int(re.findall(r"band(\d+)", folder)[0])
        print(folder, band, statsMI.shape)
        with open(os.path.join(folder,os.path.split(folder)[1]+"_globalStats.json")) as fp:
            ieeg_volt_Samp[band] = {k:np.array(v) for k, v in json.load(fp).items()}

ieeg_volt_Time = {}
for folder in glob.glob("../../NonLinearityResultsNew/FIX_iEEG_fixedTime_band?_bin*"):
    bins = int(re.findall("bin(\d+)", folder)[0])
    if os.path.isfile(os.path.join(folder,os.path.split(folder)[1]+"_globalStats.json")):
        try:
            statsMI = np.load(os.path.join(folder,f"subject00_{bins}.npy"))
        except:
            statsMI=np.array([])
        pairs = statsMI.shape[0]
        band = int(re.findall(r"band(\d+)", folder)[0])
        print(folder, band, statsMI.shape)
        with open(os.path.join(folder,os.path.split(folder)[1]+"_globalStats.json")) as fp:
            ieeg_volt_Time[band] = {k:np.array(v) for k, v in json.load(fp).items()}

In [ ]:
for i in ieeg_volt_Samp:
    ieeg_volt_Samp[i]["RNL"] = (ieeg_volt_Samp[i]["totalMI"]-ieeg_volt_Samp[i]["gaussMI"])/ieeg_volt_Samp[i]["totalMI"]
    ieeg_volt_Samp[i]["RNLshadow"] = (ieeg_volt_Samp[i]["totalMIshadow"]-ieeg_volt_Samp[i]["gaussMIshadow"])/ieeg_volt_Samp[i]["totalMIshadow"]
    ieeg_volt_Time[i]["RNL"] = (ieeg_volt_Time[i]["totalMI"]-ieeg_volt_Time[i]["gaussMI"])/ieeg_volt_Time[i]["totalMI"]
    ieeg_volt_Time[i]["RNLshadow"] = (ieeg_volt_Time[i]["totalMIshadow"]-ieeg_volt_Time[i]["gaussMIshadow"])/ieeg_volt_Time[i]["totalMIshadow"]

In [ ]:
STRETCH = 1.5
x=sorted(ieeg_volt_Samp.keys())
basepos = np.arange(len(x))*STRETCH
fig, ax = plt.subplots(1,2,figsize=(9,5), sharey=True)

for i, extra in enumerate(["","shadow"]):
    plt.sca(ax[i])
    pos = basepos-0.33
    y=[ieeg_volt_Time[i]["RNL"+extra] for i in x]
    plt.boxplot(y,positions=pos, widths=0.6,boxprops={"color":"blue"}, whiskerprops={"color":"blue"}, flierprops={"markeredgecolor":"blue", "marker":"+"}, medianprops={"lw":1.6,"color":"blue"}, capprops={"color":"blue"})

    pos = basepos+0.33
    y=[ieeg_volt_Samp[i]["RNL"+extra] for i in x]
    plt.boxplot(y,positions=pos, widths=0.6,boxprops={"color":"darkorange"}, whiskerprops={"color":"darkorange"}, flierprops={"markeredgecolor":"darkorange", "marker":"+"}, medianprops={"lw":1.6,"color":"darkorange"}, capprops={"color":"darkorange"})

    p1= plt.hlines(0.,basepos[0]-0.6, basepos[-1]+0.6, "r", "solid", lw=1, label="%", zorder=1)
    p2= plt.hlines(0.1,basepos[0]-0.6, basepos[-1]+0.6, "r", "dotted", lw=1, label="10%", zorder=1)

    plt.ylabel(f"relative difference in {extra} MI")
    plt.xlabel("Band")
    plt.xticks(ticks=basepos, labels=band_names)
    plt.title("Shadow" if extra else "True")
ax[1].yaxis.set_label_position("right")
ax[1].yaxis.tick_right()
plt.legend([Patch(facecolor="white", edgecolor="blue"), Patch(facecolor="white", edgecolor="darkorange"), p1, p2], ["Fixed time", "Fixed #samples", r"0", r"10%"], loc=0)
plt.suptitle("18 subjects $-$ Stereo Voltage")
plt.show()

In [ ]:
#tab20
avTrueRNL_iEEG_volt_samp = np.mean([ieeg_volt_Samp[i]["RNL"] for i in x], axis=1)
avShadRNL_iEEG_volt_samp = np.mean([ieeg_volt_Samp[i]["RNLshadow"] for i in x], axis=1)
avTrueRNL_iEEG_volt_time = np.mean([ieeg_volt_Time[i]["RNL"] for i in x], axis=1)
avShadRNL_iEEG_volt_time = np.mean([ieeg_volt_Time[i]["RNLshadow"] for i in x], axis=1)
palette = ListedColormap(plt.cm.tab20.colors[:-2])

fig, ax = plt.subplots(1,2,figsize=(9,5))
plt.sca(ax[0])
#A[sig_diff_reg>0.05]=np.nan
plt.scatter(avTrueRNL_iEEG_volt_samp, avShadRNL_iEEG_volt_samp,c=np.arange(0,18,2),cmap=palette, vmax=18, vmin=0, label="Fixed Samples")
plt.scatter(avTrueRNL_iEEG_volt_time, avShadRNL_iEEG_volt_time,c=np.arange(1,18,2),cmap=palette, vmax=18, vmin=0, label="Fixed Time")
cb = plt.colorbar()
cb.ax.set_ylabel('# regions', rotation=90)
cb.ax.get_yaxis().set_ticks(np.arange(1,18,2), band_names)
pts = [max(plt.xlim()[0], plt.ylim()[0]),min(plt.xlim()[1], plt.ylim()[1])]
plt.xlim(plt.xlim())
plt.ylim(plt.ylim())
plt.plot(pts, pts, ":k", lw=1)
plt.xlabel("Data RNL")
plt.ylabel("Shadow RNL")
plt.legend(loc=0, frameon=False)

plt.sca(ax[1])
plt.scatter(np.arange(0,9), avTrueRNL_iEEG_volt_samp, color=plt.cm.tab20.colors[0], label="True")
plt.scatter(np.arange(0,9), avShadRNL_iEEG_volt_samp, color=plt.cm.tab20.colors[2], label="Shadow")
plt.scatter(np.arange(0,9), avTrueRNL_iEEG_volt_samp-avShadRNL_iEEG_volt_samp, color=plt.cm.tab20.colors[4], label="Difference", marker="+")
plt.scatter(np.arange(0,9), avTrueRNL_iEEG_volt_time, color=plt.cm.tab20.colors[1], label="True")
plt.scatter(np.arange(0,9), avShadRNL_iEEG_volt_time, color=plt.cm.tab20.colors[3], label="Shadow")
plt.scatter(np.arange(0,9), avTrueRNL_iEEG_volt_time-avShadRNL_iEEG_volt_time, color=plt.cm.tab20.colors[5], label="Difference", marker="+")
# plt.xscale("log")
plt.xlabel('# regions')
plt.ylabel('RNL')
plt.legend(bbox_to_anchor=(1.2, 0.5), loc="center left", frameon=False)
plt.xticks(np.arange(0,9),band_names)
ax[1].yaxis.set_label_position("right")
ax[1].yaxis.tick_right()
plt.suptitle("Relationship between true and shadow $-$ Stereo Voltage")
plt.show()

In [ ]:
palette = ListedColormap(plt.cm.tab20.colors[:-2])

fig, ax = plt.subplots(1,2,figsize=(13,7))
for ax_n, pack in enumerate([("Samples", ieeg_volt_Samp), ("Time", ieeg_volt_Time)]):
    kind, data = pack
    plt.sca(ax[ax_n])
    for i in range(1,10):
        plt.scatter(data[i]["RNL"], data[i]["RNLshadow"],c=np.full(18, i*2+(kind=="Time")),cmap=palette, vmax=18, vmin=0, s=5)
    ax[ax_n].set_aspect(1)
    pts = [max(plt.xlim()[0], plt.ylim()[0]),min(plt.xlim()[1], plt.ylim()[1])]
    plt.xlim(plt.xlim())
    plt.ylim(plt.ylim())
    plt.plot(pts, pts, ":k", lw=1)
    plt.xlabel("Data RNL")
    plt.ylabel("Shadow RNL")
    plt.title(kind)
plt.show()


palette = plt.cm.magma
palette.set_bad(color="black") # Bad values (i.e., masked, set to black 0.8
palette.set_under(color="grey") # Bad values (i.e., masked, set to black 0.8
palette.set_over(color="grey") # Bad values (i.e., masked, set to black 0.8
for ax_n, pack in enumerate([("Samples", ieeg_volt_Samp, "o"), ("Time", ieeg_volt_Time, "^")]):
    kind, data, marker = pack
    corr_ieeg = [pearsonr(data[i]["RNL"], data[i]["RNLshadow"]) for i in range(1,10)]
    plt.scatter(np.arange(0,9), [q[0] for q in corr_ieeg],c=[q[1] for q in corr_ieeg],cmap=palette, vmax=0.05, vmin=0, marker=marker, label=kind)
plt.xticks(np.arange(0,9),band_names)
plt.legend(loc=0)
plt.show()

## spikes

# Localisation of non-linearity

## fMRI

## EEG

## iEEG

## spikes

# Sources of non-linearity

## fMRI

## EEG

## iEEG

## spikes